## Notebook for generating and getting statistical summaries of main variables in means testing form

This notebook carries out a series of processing steps in order to create proper training data to reconstruct SHA's means testing model. Aligning variables with those listed in SHA's FOI response, including dummifying, calculating new variables, and cleaning others.

Author: Purity Mukami

In [1]:
import pandas as pd
import numpy as np 
from pathlib import Path
import sys 


PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

from config import paths 
from util.normalize_column_name import normalize_column_name

### Reading in the all the relevant datasets

KNBS splits datasets into different categories, all of which can be joined via a household ID. Our wrangling scripts perform a series of preprocessing steps 

In [2]:
df_individual_microdata = pd.read_csv(paths.INTERMEDIATE_DATA / "knbs_kchs_2021/kchs_2021_individuals.csv", low_memory=False,index_col=0)
df_household_microdata = pd.read_csv(paths.INTERMEDIATE_DATA / "knbs_kchs_2021/kchs_2021_households.csv", low_memory=False,index_col=0)
df_nonfood_microdata = pd.read_csv(paths.INTERMEDIATE_DATA / "knbs_kchs_2021/kchs_2021_nonfood.csv", low_memory=False,index_col=0)
df_nonfoodknp_microdata = pd.read_csv(paths.INTERMEDIATE_DATA / "knbs_kchs_2021/kchs_2021_assets.csv", low_memory=False,index_col=0)
df_consumption_microdata = pd.read_csv(paths.INTERMEDIATE_DATA / "knbs_kchs_2021/kchs_2021_consumption.csv", low_memory=False,index_col=0)

###  Computing  the variables matching with FIOA Response PMT questions from the  kchs_21 individual dataset 
 
Unlike all the other datasets, which exist at the household level, the individual dataset as at the individual level. Varoius PMT variables, such as *mm_age* are aggregates of individual-level characteristics, while others, like *education_level* are based on the characteristics of the household head. 

Below, we calculate these various variables

#### Household composition variables 

In [3]:
"""
HOUSEHOLD-WIDE AGGREGATES from the individual level data
We use .transform() to calculate values across the whole household group instantly
Also we will Use transform('mean') on the masks grouped by hhid. This calculates the household average and maps it back to every row automatically
"""
df = df_individual_microdata.copy()
hh_groups = df.groupby("hhid")

# Demographic proportions (Age)
df["prop_over65"] = hh_groups["age_years"].transform(lambda x: (x >= 65).mean())
df["prop_under5"] = hh_groups["age_years"].transform(lambda x: (x <= 5).mean())
df["mm_age"] = np.log(hh_groups["age_years"].transform("sum"))
df['under_5_count'] = hh_groups["age_years"].transform(lambda x: (x <= 5).sum())
df['5_to_14_count'] = hh_groups["age_years"].transform(lambda x: x.between(5, 14).sum())

# Demographic proportions (M/W 18-64)
is_m1864 = (df["sex"] == "Male") & df["age_years"].between(18, 64)
is_w1864 = (df["sex"] == "Female") & df["age_years"].between(18, 64)

# Household-wide proportion using transform directly on the mask
df["prop_m1864"] = is_m1864.groupby(df["hhid"]).transform("mean")
df["prop_w1864"] = is_w1864.groupby(df["hhid"]).transform("mean")
df['num_children'] = hh_groups["age_years"].transform(lambda x: x.between(0, 17).sum())

#### Household head variables 

In [4]:
""" 
HOUSEHOLD HEAD VARIABLES 
Identify the Head row
"""
is_head = df["relationship_to_household_head"] == 'HEAD'

# Creating the occupation/status variables specifically from the Head's data. 
df.loc[is_head, "d01_own_account"] = np.where(
    df.loc[is_head, "activity_status_time_primary"] == "OWN-ACCOUNT WORKER", "Yes", "No"
)
df.loc[is_head, "d01_wage"] = np.where(
    df.loc[is_head, "activity_status_time_primary"] == "PAID EMPLOYEE (OUTSIDE HH )", "Yes", "No"
)
df.loc[is_head, "d01_own_frm"] = np.where(
    df.loc[is_head, "main_employer_primary_job"] == "SELF EMPLOYED SMALL SCALE AGRICULTURE", "Yes", "No"
)

# We extract the Head's specific values into a mapping dictionary.
head_rows = df[is_head].set_index('hhid')

df.loc[is_head, "hhhead_sex"] = df.loc[is_head, "hhid"].map(head_rows['sex'])
df.loc[is_head, "age_household_head"] = df.loc[is_head, "hhid"].map(head_rows['age_years'])
df.loc[is_head, "hhhead_marital"] = df.loc[is_head, "hhid"].map(head_rows['marital_status'])
df.loc[is_head, "hhhead_education"] = df.loc[is_head, "hhid"].map(head_rows['highest_grade_completed'])

# worked_past_week: True if head did any form of work in the past week,
# has a job to return to, is a contributing family worker,
# has an active employment status, or has recorded working hours.
# We overwrite worked_past_week so the PMT variable mapping picks it up correctly.
work_cols = [
    'worked_past_week',
    'worked_non_farm_activity_past_week',
    'worked_farm_work_past_week',
    'helped_non_farm_business_past_week',
    'helped_agricultural_past_week',
    'worked_intern_apprentice_past_week',
]
worked_last_week = (head_rows[work_cols] == 'Yes').any(axis=1)

has_job_to_return = (
    (head_rows['has_blank_to_return_to_non_farm_business'] == 'Yes') |
    (head_rows['has_blank_to_return_to_farming'] == 'Yes')
)

contributing_family_worker = (
    head_rows['activity_status_time_primary'] == 'CONTRIBUTING FAMILY WORKER'
)

employed_status = head_rows['activity_status_time_primary'].isin([
    'PAID EMPLOYEE (OUTSIDE HH )',
    'PAID EMPLOYEE ( WITHIN HH )',
    'WORKING EMPLOYER',
    'OWN-ACCOUNT WORKER',
])

worked_hours = head_rows['hours_worked_last_7_days'].fillna(0) > 0

head_rows['worked_past_week'] = (
    worked_last_week | has_job_to_return | contributing_family_worker | employed_status | worked_hours
).map({True: 'Yes', False: 'No'})

df.loc[is_head, "worked_past_week"] = df.loc[is_head, "hhid"].map(head_rows['worked_past_week'])

# FINAL STEP: CREATE THE SEPARATE HOUSEHOLD DATAFRAME 
df_individual_microdata = df[is_head].copy().reset_index(drop=True)

#### Adding the "occupation" variable to the above filtered dataset 

The PMT model contains a complicated variable on occupation, which based on the possible responses is actually a combination of various other fields in the labor section of the individual household data. The criterion for this can be seen below. 

In [5]:
#Picking up from the previous filtered dataset we start by defining the LOGICAL CONDITIONS (9 total)
final_individual_microdata = df_individual_microdata 


Criteria = [
    # 1. Public sector
    final_individual_microdata["main_employer_primary_job"].str.contains("Government|school", case=False, na=False),
    
    # 2. Farmer
    final_individual_microdata["primary_occupation_description"].str.contains("farmer|farming|farm|SUBSISTENCE", case=False, na=False)|
    (final_individual_microdata["main_employer_primary_job"] == "SELF EMPLOYED SMALL SCALE AGRICULTURE"),

    # 3. Private sector
    final_individual_microdata["main_employer_primary_job"].str.contains("PRIVATE SECTOR ENTERPRISE|LOCAL NGO/CBO|INTERNATIONAL ORGANIZATIONS/NGO|FAITH BASED ORGANIZATION", case=False, na=False ),
    
    # 4. Artisan/self-employed (7 is the code for craft and related trades workers, which includes artisans, tailors, carpenters, mechanics, welders, plumbers, painters, weavers etc.)
    (final_individual_microdata["main_employer_primary_job"].str.contains("SELF EMPLOYED -  INFORMAL|SELF EMPLOYED- FORMAL", case=False, na=False) | 
    final_individual_microdata["primary_occupation_knocs"].str.startswith("7", na=False)),
    
    # 5. Housewife
    ((final_individual_microdata["main_reason_not_looking_for_work"] == "HOME MAKER (HOUSEWIFE/FAMILY RESPONSIBILITIES)") | 
    (final_individual_microdata["main_status_last_year"] == "HOUSEWIFE/FAMILY RESPONSIBILITIES")),

    # 6. Housekeeper
    final_individual_microdata["primary_occupation_knocs"].str.contains("Cleaners Launders and Domestic Workers|" \
    "House Stewards and Housekeepers", case=False, na=False ),

    # 7. Casual labour
    (final_individual_microdata["primary_activity_working_pattern"] == "CASUAL WORKER")|
    (final_individual_microdata["primary_occupation_industry"] == "CASUAL LABOURER"),

    # 8. Retired — must come before Business lady/man, whose broad condition would otherwise absorb retired people
    (final_individual_microdata["main_reason_not_looking_for_work"] == "RETIRED")|
    (final_individual_microdata["main_status_last_year"] == "RETIRED/INCOME RECIPIENT"),
    
    # 9. Business lady/man (self-employed who are not farmers, artisans or casual workers)
    # AND (not OR) ensures we only match people who are self-employed but lack a craft/trade knocs code.
    (final_individual_microdata["main_employer_primary_job"].str.contains("SELF EMPLOYED -  INFORMAL|SELF EMPLOYED- FORMAL", case=False, na=False) & 
    ~final_individual_microdata["primary_occupation_knocs"].str.startswith("7", na=False)),
]

category = [
    "Public sector worker",
    "Farmer",
    "Private sector worker",
    "Artisan/self-employed",
    "Housewife",
    "Housekeeper",
    "Casual labour",
    "Retired",
    "Business lady/man",
]

final_individual_microdata["occupation"] = np.select(Criteria, category, default="Other")


#### Clean household head education level 

The education level of the household head needs to be aligned with the responses for the PMT form. 

In [6]:
# Define the mapping based on the FIOA response categories
education_map = {
    # Primary
    'STANDARD/GRADE 1': 'Primary', 'STANDARD/GRADE 2': 'Primary', 
    'STANDARD/GRADE 3': 'Primary', 'STANDARD/GRADE 4': 'Primary',
    'STANDARD/GRADE 5': 'Primary', 'STANDARD/GRADE 6': 'Primary',
    'STANDARD/GRADE 7': 'Primary', 'STANDARD/GRADE 8': 'Primary',
    
    # Secondary Level
    'FORM 1/GRADE 9': 'Secondary level', 'FORM 2//GRADE 10': 'Secondary level',
    'FORM 3/GRADE 11': 'Secondary level', 'FORM 4/GRADE 12': 'Secondary level',
    'FORM 5': 'Secondary level', 'FORM 6': 'Secondary level',
    'ADULT SECONDARY EDUCATION': 'Secondary level',
    
    # Middle level college
    'DIPLOMA/CERTIFICATE YEAR 1': 'Middle level college (certificate/diploma)',
    'DIPLOMA/CERTIFICATE YEAR 2': 'Middle level college (certificate/diploma)',
    'DIPLOMA YEAR 3': 'Middle level college (certificate/diploma)',
    'HIGHER NATIONAL DIPLOMA': 'Middle level college (certificate/diploma)',
    
    # University Undergraduate
    'UNDERGRADUATE YEAR 1': 'University(Undergraduate)', 'UNDERGRADUATE YEAR 2': 'University(Undergraduate)',
    'UNDERGRADUATE YEAR 3': 'University(Undergraduate)', 'UNDERGRADUATE YEAR 4': 'University(Undergraduate)',
    'UNDERGRADUATE YEAR 5': 'University(Undergraduate)', 'UNDERGRADUATE YEAR 6': 'University(Undergraduate)',
    
    # University Post Graduate
    'MASTERS YEAR 1': 'University (Post Graduate)', 'MASTERS YEAR 2': 'University (Post Graduate)',
    'PHD YEAR 1': 'University (Post Graduate)', 'PHD YEAR 2': 'University (Post Graduate)',
    'PHD YEAR 3': 'University (Post Graduate)',
    
    # Vocational
    'VOCATIONAL TRAINING YEAR 1': 'Vocational', 'VOCATIONAL TRAINING YEAR 2': 'Vocational',
    
    # Informal / Madrasa
    'MADRASSA/DUKSIS': 'Informal education (madrasa/adult basic)',
    'ADULT BASIC EDUCATION': 'Informal education (madrasa/adult basic)',
    
    # Pre-Primary
    'BABY CLASS /KINDERGARTEN 1': 'Pre-primary', 'NURSERY/KINDERGARTEN 2': 'Pre-primary',
    'PRE UNIT/KINDERGARTEN 3': 'Pre-primary',
    
    # None / Don't Know
    'NOT STATED/DK': 'Dont know', 'NOT APPLICABLE': 'None'
}

In [7]:
# strip spaces and convert empty strings to NaN
final_individual_microdata['hhhead_education_clean'] = final_individual_microdata['hhhead_education'].astype(str).str.strip().replace('', np.nan)

# Apply the mapping
final_individual_microdata['education_level'] = (
    final_individual_microdata['hhhead_education_clean']
    .map(education_map)
    .fillna('Dont know')
)
final_individual_microdata.drop(columns=['hhhead_education_clean'], inplace=True)
final_individual_microdata['education_level'].value_counts()

education_level
Primary                                       7427
Secondary level                               4262
Dont know                                     2809
Middle level college (certificate/diploma)    1484
University(Undergraduate)                      659
Vocational                                     184
University (Post Graduate)                      96
Informal education (madrasa/adult basic)        90
Pre-primary                                     25
None                                             6
Name: count, dtype: int64

### Merging all the  datsets at household level

This is based on 'hhid'

In [8]:
original_length = len(df_household_microdata)

merged_df = pd.merge(final_individual_microdata, df_household_microdata, on= 'hhid', how= 'right', suffixes=(None, '_household'))
merged_df = pd.merge(merged_df, df_nonfood_microdata, on='hhid', how = 'left', suffixes=(None, '_nonfood'))
merged_df = pd.merge(merged_df, df_nonfoodknp_microdata, on='hhid', how = 'left', suffixes=(None, '_nonfoodknp'))
merged_df = pd.merge(merged_df, df_consumption_microdata, on='hhid', how='left', suffixes=(None, '_consumption'))

merged_df = merged_df.reset_index(drop=True)

# Assert that merge has occured correctly. 
assert len(merged_df) == original_length, "Merging altered the number of households!"

merged_df.head()

,strid,hhid,respondent_id,county,urban_rural_classification,stratum,relationship_to_household_head,sex,age_years,age_months,...,padqexp,padqrent,padqeduc,pl_food,pl_abs,food_poor,abs_poor,hc_poor,adq_scale_consumption,unique_id_consumption
0,1.0,1.0,1,Mombasa,Urban,1,HEAD,Male,38.0,NaN,...,20739.951,3724.9453,0.0000,2904.9536,7192.9536,0.0,0.0,0.0,1.0,1_1
1,2.0,2.0,1,Mombasa,Urban,1,HEAD,Male,41.0,NaN,...,14814.364,1751.2264,0.0000,2904.9536,7192.9536,0.0,0.0,0.0,1.0,1_2
2,3.0,3.0,1,Mombasa,Urban,1,HEAD,Female,25.0,NaN,...,23199.200,1945.2678,3751.5886,2904.9536,7192.9536,0.0,0.0,0.0,2.0,1_3
3,4.0,4.0,1,Mombasa,Urban,1,HEAD,Male,37.0,NaN,...,15073.915,3086.4850,0.0000,2904.9536,7192.9536,0.0,0.0,0.0,2.0,1_4
4,5.0,5.0,1,Mombasa,Urban,1,HEAD,Male,52.0,NaN,...,11713.136,4544.5044,0.0000,2904.9536,7192.9536,0.0,0.0,0.0,2.0,1_5


### Creating a new variable called "access_electricity" in the merged dataset. 
-- This is because there are two variables that can help give a clearer picture on who has access to electricity as opposed to those who only pay for it. 
-- We check whether a household has paid for electricity (nfcons) or uses electiricty for Lighting. 


In [9]:
# Use np.where to check if either column indicates electricity access
merged_df["access_electricity"] = np.where(
    (merged_df["electricity"] > 0) | 
    (merged_df["main_energy_source_lighting"] == "Electricity"),
    
    "Yes",
    "No"
)

#### Normalize asset columns 

In the data, asset columns distinguish vs recently purchased and own, but no such distinction is made in PMT form 

In [10]:
# Find columns that have 'Purchased' or 'Own' in any of their rows
target_cols = [col for col in merged_df.columns if merged_df[col].isin(['Purchased', 'Own']).any()]
target_cols[:10]

['owns_curtains_and_accessories',
 'owns_blankets',
 'owns_pillows',
 'owns_mattresses',
 'owns_bed_sheets_bed_covers_pillow_cases',
 'owns_table_cloth_mats',
 'owns_mosquito_net',
 'owns_towels',
 'owns_carpets_and_mats',
 'owns_other_coverings']

In [11]:
for col in target_cols:
# If the value is 'Purchased' or 'Own', make it 'Yes', else 'No'
    merged_df[col] = np.where(merged_df[col].isin(['Purchased', 'Own']), 'Yes', 'No')

### Map the variables provided by SHA to those within the KNBS survey data. 


In [12]:
# We start by reading the tab that has all the mapping in a xls file
kchs_2021_PMT_VARIABLES = pd.read_excel(paths.CONFIG / 'pmt_variables.xlsx', sheet_name='kchs_2021_pmt_variables')

kchs_2021_PMT_VARIABLES.head()

,original_name,Calculated_name_after_Transformation,original_name_after_Transformation,newFIOA_name,In_Merged_name,In_Training_Data
0,hhid,hhid,hhid,hhid,hhid,hhid
1,county,county,county,County,county,County
2,total_persons_household,total_persons_household,total_persons_household,household_size,total_persons_household,household_size
3,NaN,NaN,NaN,mm_age,mm_age,mm_age
4,NaN,NaN,NaN,prop_over65,prop_over65,prop_over65


In [13]:
# Create mapping and rename 
pmt_rename_dict = dict(zip(kchs_2021_PMT_VARIABLES['original_name_after_Transformation'], kchs_2021_PMT_VARIABLES['newFIOA_name']))
merged_df.rename(columns=pmt_rename_dict, inplace=True)

#### We now have the full version of our training set ready 


In [14]:
training_data_all = merged_df

### Create a version of training set with only variables directly on the PMT. 

In [15]:
foi_pmt_variables = kchs_2021_PMT_VARIABLES[['newFIOA_name']]

In [16]:
# Create the list from Excel and add our two extra variables in one go
foi_pmt_variables = kchs_2021_PMT_VARIABLES['newFIOA_name'].tolist() + ['urban_rural_classification', 'padqexp'] # We add these two 

foi_pmt_variables_in_merged = [col for col in foi_pmt_variables if col in merged_df.columns]

training_data_filtered = merged_df[foi_pmt_variables_in_merged]

### Dummify categorical variables  

In [17]:
categorical_cols = training_data_filtered.select_dtypes(include='object').columns.tolist()

# Create a new list that EXCLUDES the classification column
cols_to_dummy = [col for col in categorical_cols if col != 'urban_rural_classification']

# Now run get_dummies only on the filtered list
training_data_filtered_with_dummies = pd.get_dummies(training_data_filtered, columns=cols_to_dummy, drop_first=True)

#### Convert True and False columns in the merged file to 0s and 1s

In [18]:
 
bool_cols = training_data_filtered_with_dummies.select_dtypes(include='bool').columns 
training_data_filtered_with_dummies[bool_cols] = training_data_filtered_with_dummies[bool_cols].astype(int)

#### Normalize column names and drop rows with N/As 

In [19]:
training_data_filtered_with_dummies.columns = training_data_filtered_with_dummies.columns.map(normalize_column_name)

length_before = len(training_data_filtered_with_dummies)
training_data_filtered_with_dummies = training_data_filtered_with_dummies.dropna()
lenght_after = len(training_data_filtered_with_dummies)

print(f"Dropped {length_before - lenght_after} rows due to missing values.")

Dropped 79 rows due to missing values.


In [20]:
training_data_filtered_with_dummies.head()

,hhid,household_size,mm_age,prop_over65,prop_under5,prop_m1864,prop_w1864,nr_rooms,urban_rural_classification,padqexp,...,d1_own_account_yes,d1_own_frm_yes,occupation_casual_labour,occupation_farmer,occupation_housekeeper,occupation_housewife,occupation_other,occupation_private_sector_worker,occupation_public_sector_worker,occupation_retired
0,1.0,1.0,3.637586,0.0,0.0,1.0,0.0,1,Urban,20739.951,...,1,0,0,0,0,0,0,0,0,0
1,2.0,1.0,3.713572,0.0,0.0,1.0,0.0,1,Urban,14814.364,...,1,0,0,0,0,0,0,0,0,0
2,3.0,2.0,3.931826,0.0,0.0,0.5,0.5,1,Urban,23199.200,...,0,0,0,0,0,0,1,0,0,0
3,4.0,2.0,4.127134,0.0,0.0,1.0,0.0,1,Urban,15073.915,...,0,0,0,0,0,0,0,0,1,0
4,5.0,2.0,4.343805,0.0,0.0,0.5,0.5,1,Urban,11713.136,...,0,0,0,0,0,0,1,0,0,0


### Create our log consumption, which will be our label 

In [21]:
# Calculate log consumption 
training_data_filtered_with_dummies['actual_consumption_log'] = np.log(training_data_filtered['padqexp'])
training_data_filtered_with_dummies = training_data_filtered_with_dummies.drop(columns=['padqexp'])

/var/folders/hp/kn6pmy_14nv6djdz1m3qrg200000gn/T/ipykernel_9403/3998986374.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  training_data_filtered_with_dummies['actual_consumption_log'] = np.log(training_data_filtered['padqexp'])


### Save the filtered and unfiltered training data 

In [22]:
training_data_filtered_with_dummies.to_csv(paths.PROCESSED_DATA / "td_kchs_2021_filtered.csv", index=False)

training_data_all = training_data_all.merge(training_data_filtered_with_dummies, on='hhid', how='left', suffixes=(None, '_all'))
training_data_all.to_csv(paths.PROCESSED_DATA / "td_kchs_2021_all.csv", index=False)
